In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import  create_engine
import seaborn as sns
import time

In [2]:
database = 'Sales_DataWareHouse'
server = r'OMKAR\SQLEXPRESS'
conn_url = f'mssql+pyodbc://@{server}/{database}?driver=ODBC+DRIVER+17+for+SQL+server&Trustred_Connection = YES'
engine = create_engine(conn_url)

In [3]:
db_tables = ['gold.dim_customer', 'gold.dim_product', 'gold.dim_orders', 'gold.fact_sales']
df = {}
for i in db_tables:
    query = f'SELECT * FROM {i};'
    df[i] = pd.read_sql_query(query, con = engine)

In [4]:
print('=' * 100)
print('*' * 39 + ' Data Quality Report ' + '*'* 40 )
for i in db_tables:
    print('=' * 38 + f' {i} ' + '='*38)
    print(f'{df[i].shape[0]} Rows and {df[i].shape[1]} Columns')
    print('+'*50)
    duplicated = df[i].duplicated().sum()
    print(f'There are {duplicated} duplicated rows.')
    print('+'*50)
    print(f'{df[i].info()}')
    print('+'*50)
    print('-' * 38 + 'Statistical Analysis' + '-'*38)
    print(df[i].describe().T)
    missing_summary={
    "Missing Count": df[i].isna().sum(),
    "Missing Percentage" :df[i].isnull().mean() * 100
    }
    print('-' * 38 + 'Missing Analysis' + '-'*38)
    print(f'Missing Count:\n{missing_summary["Missing Count"]}\n\nMissing Percentage:\n{missing_summary["Missing Percentage"]}')
    print('-' * 50)
    print(f'Numbers of Unique Fields in {i}:\n{df[i].nunique()}')
    print('-' * 50)

    print('\n\n')

*************************************** Data Quality Report ****************************************
====================================== gold.dim_customer ======================================
40000 Rows and 16 Columns
++++++++++++++++++++++++++++++++++++++++++++++++++
There are 0 duplicated rows.
++++++++++++++++++++++++++++++++++++++++++++++++++
<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          40000 non-null  str    
 1   customer_first_name  40000 non-null  str    
 2   customer_last_name   40000 non-null  str    
 3   gender               40000 non-null  str    
 4   age                  40000 non-null  int64  
 5   age_group            40000 non-null  str    
 6   dob                  40000 non-null  object 
 7   email                40000 non-null  str    
 8   phone                40000 non-null  str   

## Datetime Conversion

In [5]:
df['gold.dim_customer']['dob'] = pd.to_datetime(df['gold.dim_customer']['dob'])
df['gold.dim_customer']['registration_date'] = pd.to_datetime(df['gold.dim_customer']['registration_date'])

df['gold.dim_orders']['order_date'] = pd.to_datetime(df['gold.dim_orders']['order_date'])
df['gold.dim_orders']['delivery_date'] = pd.to_datetime(df['gold.dim_orders']['delivery_date'])

### Table Extraction.

In [6]:
customer_df = df['gold.dim_customer']
product_df = df['gold.dim_product']
orders_df = df['gold.dim_orders']
sales_df = df['gold.fact_sales']

In [7]:
sales_data_df = sales_df.merge(orders_df, how= 'left', on = 'order_id')
sales_data_df = sales_data_df.merge(product_df, how = 'left', on = 'product_id')
sales_data_df = sales_data_df.merge(customer_df, how = 'left', on = 'customer_id')

In [8]:
sales_data_df['customer_name'] = sales_data_df['customer_first_name'].str.cat(sales_data_df['customer_last_name'], sep = " ")

In [9]:
sales_data_df.rename(columns={'city_y':'customer_city','state_y':'customer_state','quantity':'order_quantity', 'city_x': 'order_city', 'state_x': 'order_state'},inplace=True)

In [10]:
sales_data_df = sales_data_df[['order_id', 'customer_id', 'product_id', 'customer_name', 'gender', 'age','age_group', 'dob', 'email', 'phone', 'customer_city', 'customer_state', 'pinecode','registration_date', 'tier', 'total_orders', 'total_spent','product_name', 'category', 'brand', 'price','discount_percent', 'discount_amount', 'selling_amount','stock_quantity', 'weight_kg', 'avg_rating', 'total_reviews','order_quantity', 'unit_price','order_value', 'shipping_cost', 'coupon_code', 'coupon_discount','total_amount', 'order_date', 'order_time', 'delivery_date','payment_mode', 'order_status', 'rating', 'review_text', 'order_city','order_state']]

In [12]:
sales_data_df.isnull().sum()

order_id                  0
customer_id               0
product_id                0
customer_name             0
gender                    0
age                       0
age_group                 0
dob                       0
email                     0
phone                     0
customer_city             0
customer_state            0
pinecode                  0
registration_date         0
tier                      0
total_orders              0
total_spent               0
product_name              0
category                  0
brand                     0
price                     0
discount_percent          0
discount_amount           0
selling_amount            0
stock_quantity            0
weight_kg                 0
avg_rating                0
total_reviews             0
order_quantity            0
unit_price                0
order_value               0
shipping_cost             0
coupon_code          199815
coupon_discount           0
total_amount              0
order_date          

In [13]:
sales_data_df.duplicated().sum()

np.int64(0)

In [47]:
first_sales_date = sales_data_df['order_date'].min()
latest_sales_date = sales_data_df['order_date'].max()
time = latest_sales_date - first_sales_date
data_shape = sales_data_df.shape

print(f"First order made is {first_sales_date.strftime('%d/%m/%Y')} and the last or latest order made date is {latest_sales_date.strftime('%d/%m/%Y')} so we have {time.days/365.25:.0f} Years Transaction data With {data_shape[0]} rows & {data_shape[1]} columns")

First order made is 01/06/2024 and the last or latest order made date is 30/06/2026 so we have 2 Years Transaction data With 250000 rows & 44 columns


## key profitability drivers

### Total Revenue.
#### The total revenue generated before subtracting discounts and shipping fees.

##### Q. What is Total Revenue?

In [50]:
print(f' The total revenue {sales_data_df['order_value'].sum()/1000000:.2f} M generated past 2 years.')

 The total revenue 5991.97 M generated past 2 years.


##### Q. What is the Total Revenue we made in last year from June 2024 to June 2025?

In [66]:
start_date = '2024-06-01'
end_date = '2025-06-30'
mask = (sales_data_df['order_date'] >= start_date) & (sales_data_df['order_date'] <= end_date)
filtered_df = sales_data_df.loc[mask]

first_sales_date = filtered_df['order_date'].min()
latest_sales_date = filtered_df['order_date'].max()
time = latest_sales_date - first_sales_date
data_shape = filtered_df.shape

total_revenue_2025 = filtered_df['order_value'].sum()/1000000

print(f"First order made is {first_sales_date.strftime('%d/%m/%Y')} and the last or latest order made date is {latest_sales_date.strftime('%d/%m/%Y')} so we have {time.days/365.25:.0f} Years Transaction data With {data_shape[0]} rows & {data_shape[1]} columns")
print(f'The total revenue {total_revenue_2025:.2f} M generated last years.')

First order made is 01/06/2024 and the last or latest order made date is 30/06/2025 so we have 1 Years Transaction data With 130070 rows & 44 columns
The total revenue 3113.61 M generated last years.


##### Q. What is the Total Revenue we generate this year?

In [76]:
start_date = '2025-06-01'
end_date = '2026-06-30'
mask = (sales_data_df['order_date'] >= start_date) & (sales_data_df['order_date'] <= end_date)
filtered_df = sales_data_df.loc[mask]

first_sales_date = filtered_df['order_date'].min()
latest_sales_date = filtered_df['order_date'].max()
time = latest_sales_date - first_sales_date
data_shape = filtered_df.shape

total_revenue_2026 = filtered_df['order_value'].sum()/1000000

print(f"First order made is {first_sales_date.strftime('%d/%m/%Y')} and the last or latest order made date is {latest_sales_date.strftime('%d/%m/%Y')} so we have {time.days/365.25:.0f} Years Transaction data With {data_shape[0]} rows & {data_shape[1]} columns")
print(f'The total revenue {total_revenue_2026:.2f} M generated this years.')
print(f'We made {round(total_revenue_2025 - total_revenue_2026,2)} M less total revenue from last year.')

First order made is 01/06/2025 and the last or latest order made date is 30/06/2026 so we have 1 Years Transaction data With 129849 rows & 44 columns
The total revenue 3113.01 M generated this years.
We made 0.6 M less total revenue from last year.


### Total Profit.

#### The final remaining after deducting all operating expenses, costs of goods sold, discounts, and shipping fees from total revenue

In [77]:
sales_data_df.columns

Index(['order_id', 'customer_id', 'product_id', 'customer_name', 'gender',
       'age', 'age_group', 'dob', 'email', 'phone', 'customer_city',
       'customer_state', 'pinecode', 'registration_date', 'tier',
       'total_orders', 'total_spent', 'product_name', 'category', 'brand',
       'price', 'discount_percent', 'discount_amount', 'selling_amount',
       'stock_quantity', 'weight_kg', 'avg_rating', 'total_reviews',
       'order_quantity', 'unit_price', 'order_value', 'shipping_cost',
       'coupon_code', 'coupon_discount', 'total_amount', 'order_date',
       'order_time', 'delivery_date', 'payment_mode', 'order_status', 'rating',
       'review_text', 'order_city', 'order_state'],
      dtype='str')

In [83]:
sales_data_df.to_csv('sales_data.csv')